In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn import datasets, metrics, svm
from sklearn import svm, tree, neighbors
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

In [2]:
odroid1 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid1_firefox_onscreen.csv")
odroid2 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid2_firefox_onscreen.csv")
odroid3 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid3_firefox_onscreen.csv")
odroid4 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid4_firefox_onscreen.csv")

In [3]:
rpi4a = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-4a_chrome_onscreen.csv")
rpi8a = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8a_chrome_onscreen.csv")
rpi8b = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8b_chrome_onscreen.csv")
rpi8c = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8c_chrome_onscreen.csv")
rpi8d = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8d_chrome_onscreen.csv")

In [83]:
df = pd.concat([odroid1[:1000], odroid2[:1000], odroid3[:1000], odroid4[:1000], rpi4a[:1000], rpi8a[:1000], rpi8b[:1000], rpi8c[:1000], rpi8d[:1000]])
df = df.sample(frac=1).reset_index(drop=True)

In [84]:
df.head()

,label,Feature 0,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6
0,rpi-4a,17.0,18.6,780.8,422.9,17.7,207.6,19.3
1,rpi-8a,230.8,231.2,233.3,433.3,16.8,233.2,16.5
2,rpi-8c,16.7,25.6,39.6,73.8,23.1,100.2,959.4
3,rpi-8b,20.7,3.7,427.8,232.6,433.6,16.5,233.0
4,odroid2,165.0,177.0,155.0,170.0,169.0,10.0,166.0


In [85]:
encoder = LabelEncoder()

encoded_labels = pd.DataFrame(encoder.fit_transform(df['label']))

In [86]:
label_map = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

In [87]:
label_map 


{'odroid1': 0,
 'odroid2': 1,
 'odroid3': 2,
 'odroid4': 3,
 'rpi-4a': 4,
 'rpi-8a': 5,
 'rpi-8b': 6,
 'rpi-8c': 7,
 'rpi-8d': 8}

In [88]:
df = df.drop(['label'], axis=1)

In [89]:
X_train, X_test, y_train, y_test = train_test_split(
    df, encoded_labels, test_size=0.25, shuffle=False
)

In [90]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# clf = Pipeline(
#     steps=[("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=9))]
# )

In [91]:
# Choose a range of k values to try out (for example, from 1 to 25)
param_grid = {'n_neighbors': [i for i in range(1, 26)]}

# Use GridSearchCV to find the best value of k
knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train[0])

# Print the best parameters found by grid search
print("Best hyperparameters: ", grid_search.best_params_)

# Evaluate the model on the test set using the best k value
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_test)
print(classification_report(y_test, y_pred))

Best hyperparameters:  {'n_neighbors': 1}
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       273
           1       0.96      1.00      0.98       233
           2       0.94      0.93      0.93       266
           3       0.99      0.98      0.99       240
           4       0.87      0.85      0.86       245
           5       0.75      0.78      0.76       247
           6       0.80      0.79      0.79       274
           7       0.67      0.65      0.66       241
           8       0.68      0.70      0.69       231

    accuracy                           0.85      2250
   macro avg       0.85      0.85      0.85      2250
weighted avg       0.85      0.85      0.85      2250

